In [ ]:
!pip install quimb mps-to-circuit qiskit qiskit_aer

In [ ]:
from builder import *
from scipy.linalg import expm
from qiskit.circuit.library import UnitaryGate
from mps_to_circuit import mps_to_circuit
from qiskit import transpile
from qiskit.circuit import IfElseOp, ControlFlowOp
import csv

N = 12
J1 = 1.0
J2 = 0.7
TROTTER_SWEEP = [1, 2, 4, 8, 16]
SWEEP_L = 1         # trial-circuit layer count to use
COUNT_RESOURCES = True              # transpile at opt level 3 can be slow for large steps
BASIS = ("h", "s", "cx", "rz")

H_mpo = MPO_ham_j1j2(N, j1=J1, j2=J2, cyclic=False)
res = get_energies(H_mpo, N=N, j1=J1, j2=J2, run_ed_check=True)
E0_dmrg, psi0_mps, ed = res["E0"], res["psi0"], res["ed"]
if ed is None:
    raise RuntimeError("ED cross-check didn't run (check ED_CHECK_MAX_N / N).")
psi0_ed, E0_ed = ed["psi0_ed"], ed["E0_ed"]



H_qk = j1j2_hamiltonian(N, j1=J1, j2=J2)
spec = get_spectrum(N=N, j1=J1, j2=J2, source="dmrg")

E0, Etop = float(np.real(spec["E0"])), float(np.real(spec["Etop"]))
H_shifted = H_qk + SparsePauliOp.from_list([("I" * N, -E0)])
H_scaled_qk = (H_shifted / (Etop - E0)).simplify()

psi0_dmrg = np.asarray(psi0_mps.to_dense()).reshape(-1)
psi0_dmrg = psi0_dmrg / np.linalg.norm(psi0_dmrg)



# --- exact-unitary pulse (reference, no Trotter) ---------------------------
_expm_cache = {}
def _U(Hm, t):
    key = round(float(t), 12)
    if key not in _expm_cache:
        _expm_cache[key] = (expm(1j * t * Hm), expm(-1j * t * Hm))
    return _expm_cache[key]




def postselected_run(trial_qc, times, phases, pulse_fn, synthesize=True):
    """Apply each pulse, project ancilla -> 0, renormalize.
    synthesize=True forces PauliEvolutionGate -> actual Trotter gates before
    simulating (otherwise Statevector uses the exact matrix). Set False only
    for the UnitaryGate reference."""
    dim = 2 ** N
    sys_q, anc = list(range(N)), N
    sv = Statevector(np.concatenate([Statevector(trial_qc).data,
                                     np.zeros(dim, dtype=complex)]))
    p_succ = 1.0
    for t_i, phi_i in zip(times, phases):
        pc = QuantumCircuit(N + 1)
        pulse_fn(pc, sys_q, anc, t_i, phi_i)
        if synthesize:
            pc = transpile(pc, basis_gates=["h", "s", "x", "cx", "rz"],
                           optimization_level=0)
        sv = sv.evolve(pc)
        proj = np.asarray(sv.data)[:dim]
        p = float(np.vdot(proj, proj).real)
        p_succ *= p
        sv = Statevector(np.concatenate([proj / np.sqrt(p), np.zeros(dim, dtype=complex)]))
    return p_succ, np.asarray(sv.data)[:dim]


def state_metrics(sys_vec):
    E = float(np.real(Statevector(sys_vec).expectation_value(H_qk)))
    t = np.transpose(sys_vec.reshape([2] * N), tuple(range(N - 1, -1, -1))).reshape(-1)
    return (E,
            float(abs(np.vdot(psi0_dmrg, t)) ** 2),
            float(abs(np.vdot(psi0_ed, t)) ** 2))



def apply_pulse_exact_np(state, Hm, t_i, phi_i, dim):
    """One exact filter pulse via direct numpy block manipulation.
    No Gate.control() synthesis involved anywhere.
    state: flat complex array, length 2*dim, laid out [anc=0 block, anc=1 block]
    (ancilla = MSB, matching the rest of the script's convention)."""
    Ufwd, Ubwd = _U(Hm, t_i)                       # e^{+iHt}, e^{-iHt}, dim x dim

    s = state.reshape(2, dim)                      # s[0]=anc0 block, s[1]=anc1 block

    h = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
    rz = np.diag([np.exp(-1j * phi_i), np.exp(1j * phi_i)])
    anc_op = rz @ h                                 # H then rz(2*phi), circuit order
    s = np.einsum('ab,b...->a...', anc_op, s)

    # controlled evolution: anc=1 branch -> fwd, anc=0 branch -> bwd
    # (matches the X-sandwich trick in apply_filter_pulse)
    s = np.stack([Ubwd @ s[0], Ufwd @ s[1]])

    s = np.einsum('ab,b...->a...', h, s)            # final H on ancilla
    return s.reshape(-1)


def postselected_run_exact(trial_qc, times, phases, Hm):
    """Exact-unitary post-selected run, pure numpy -- no qiskit control()."""
    dim = 2 ** N
    psi = Statevector(trial_qc).data.astype(complex)
    state = np.concatenate([psi, np.zeros(dim, dtype=complex)])  # anc=0 block, anc=1 block

    p_succ = 1.0
    for t_i, phi_i in zip(times, phases):
        state = apply_pulse_exact_np(state, Hm, t_i, phi_i, dim)
        anc0 = state[:dim]
        p = float(np.vdot(anc0, anc0).real)
        p_succ *= p
        state = np.concatenate([anc0 / np.sqrt(p), np.zeros(dim, dtype=complex)])

    return p_succ, state[:dim]

def flatten_success_path(circ):
    """Inline every IfElseOp's true-body so the circuit has no control flow.
    For the nested `if cycle_bit == 0:` structure this is exactly the
    all-zeros (post-selected) path."""
    flat = QuantumCircuit(list(circ.qubits), list(circ.clbits))
    for inst in circ.data:
        op = inst.operation
        if isinstance(op, IfElseOp):
            body = flatten_success_path(op.blocks[0])
            flat.compose(body, qubits=list(inst.qubits),
                         clbits=list(inst.clbits), inplace=True)
        elif isinstance(op, ControlFlowOp):
            raise NotImplementedError(f"unhandled control flow: {op.name}")
        else:
            flat.append(op, inst.qubits, inst.clbits)
    return flat
    
def resource_costs(circuit, label="", flatten=False, verbose=True):
    """Transpile to BASIS and count. flatten=True for the rodeo circuit."""
    circ = flatten_success_path(circuit) if flatten else circuit
    rc = transpile(circ, basis_gates=list(BASIS) + ["measure", "reset"],
                   optimization_level=3)
    counts = rc.count_ops()
    tidy = {g: int(counts.get(g, 0)) for g in BASIS}
    mid = {g: int(counts.get(g, 0)) for g in ("measure", "reset") if g in counts}
    other = {g: int(c) for g, c in counts.items()
             if g not in BASIS and g not in ("measure", "reset", "barrier")}
    out = {"h": tidy["h"], "s": tidy["s"], "cx": tidy["cx"], "rz": tidy["rz"],
           "total": sum(tidy.values()), "depth": rc.depth(),
           "measure": mid.get("measure", 0), "reset": mid.get("reset", 0),
           "other": other}
    if verbose:
        print(f"\n--- resource costs: {label} ---")
        print("  " + "  ".join(f"{g.upper()}={tidy[g]}" for g in BASIS)
              + f"  | total={out['total']}  depth={out['depth']}"
              + (f"  | mid-circuit: {mid}" if mid else ""))
        if other:
            print(f"  [warning] non-basis ops left after transpile: {other}")
    return out

H_mpo = MPO_ham_j1j2(N, j1=J1, j2=J2)
res = get_energies(H_mpo, N=N, j1=J1, j2=J2, run_ed_check=True)
E0_dmrg, psi0_mps, ed = res["E0"], res["psi0"], res["ed"]
builder = FilterBuilder(total_time=spec["totaltime"], energies=spec["energies"])
best = builder.build(method="v3")[-1]
sw_times, sw_phases = best["times"], best["phases"]
tqc_ = mps_to_circuit(psi0_mps.arrays, method="approximate",
                          shape="lpr", num_layers=1)
if tqc_.num_qubits != N:
    raise ValueError(f"trial circuit has {tqc_.num_qubits} qubits, expected {N}")

trial_qc = tqc_





print(f"\nTrotter sweep: L={SWEEP_L}, {len(sw_times)} cycles, "
      f"t_i range [{sw_times.min():.3f}, {sw_times.max():.3f}] (scaled-H units)")

# --- exact reference --------------------------------------------------------
Hm = H_scaled_qk.to_matrix()
p_ex, sv_ex = postselected_run_exact(trial_qc, sw_times, sw_phases, Hm)
E_ex, Fd_ex, Fe_ex = state_metrics(sv_ex)
print(f"[exact]  P_succ={p_ex:.6f}  E-E0dmrg={E_ex - E0_dmrg:.3e}  "
      f"F_dmrg={Fd_ex:.6f}  F_ed={Fe_ex:.6f}")

# --- Trotter sweep ----------------------------------------------------------
trot_rows = []
for r in TROTTER_SWEEP:
    p, sv = postselected_run(
        trial_qc, sw_times, sw_phases,
        lambda qc, s, a, t, ph, r=r: apply_filter_pulse(qc, s, a, H_scaled_qk, t, ph, r))
    E, Fd, Fe = state_metrics(sv)
    # distance to the exact-filter output state (phase-insensitive)
    F_vs_exact = float(abs(np.vdot(sv_ex, sv)) ** 2)

    row = {"trotter_steps": r, "P_succ": p, "E-E0_dmrg": E - E0_dmrg,
           "F_dmrg": Fd, "F_ed": Fe, "F_vs_exact_filter": F_vs_exact,
           "dP_vs_exact": p - p_ex}

    if COUNT_RESOURCES:
        qc_r, _ = build_rodeo_circuit(H_scaled_qk, sw_times, sw_phases,
                                      trial_prep=lambda qc, s: None,
                                      trotter_steps=r)
        c = resource_costs(qc_r, label=f"filter only, {r} Trotter steps", flatten=True, verbose=False)
        row.update({"cx": c["cx"], "total": c["total"], "depth": c["depth"]})

    trot_rows.append(row)
    print(f"[r={r:>2}] P_succ={p:.6f}  E-E0={E - E0_dmrg:.3e}  F_dmrg={Fd:.6f}  "
          f"F_vs_exact={F_vs_exact:.6f}" + (f"  CX={row['cx']}  depth={row['depth']}" if COUNT_RESOURCES else ""))

with open("trotter_sweep.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(trot_rows[0].keys()))
    w.writeheader(); w.writerows(trot_rows)

# --- plot -------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rs = [r["trotter_steps"] for r in trot_rows]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].semilogy(rs, [max(1 - r["F_vs_exact_filter"], 1e-16) for r in trot_rows], "o-")
ax[0].set_xscale("log", base=2)
ax[0].set_xlabel("Trotter steps per pulse"); ax[0].set_ylabel("1 - |<exact filter|Trotter filter>|^2")
ax[0].set_title("Trotter error in filtered state")

ax[1].plot(rs, [r["F_dmrg"] for r in trot_rows], "o-", label="F vs DMRG (Trotter)")
ax[1].axhline(Fd_ex, color="C0", ls="--", alpha=0.6, label="F vs DMRG (exact)")
ax[1].set_xscale("log", base=2)
ax[1].set_xlabel("Trotter steps per pulse"); ax[1].set_ylabel("post-filter fidelity", color="C0")
axr = ax[1].twinx()
axr.plot(rs, [r["P_succ"] for r in trot_rows], "s-", color="C1", label="P_succ (Trotter)")
axr.axhline(p_ex, color="C1", ls="--", alpha=0.6, label="P_succ (exact)")
axr.set_ylabel("success probability", color="C1")
ax[1].set_title("Fidelity and success prob (dashed = exact)")
ax[1].legend(loc="lower right")
lines1, labels1 = ax[1].get_legend_handles_labels()
lines2, labels2 = axr.get_legend_handles_labels()
ax[1].legend(lines1 + lines2, labels1 + labels2, loc="lower right")


if COUNT_RESOURCES:
    ax[2].plot(rs, [r["cx"] for r in trot_rows], "o-", label="CX")
    ax[2].plot(rs, [r["depth"] for r in trot_rows], "s-", label="depth")
    #ax[2].set_xscale("log", base=2); ax[2].set_yscale("log")
    ax[2].set_xscale("log", base=2)
    ax[2].set_xlabel("Trotter steps per pulse"); ax[2].set_ylabel("count (basis h,s,cx,rz)")
    ax[2].set_title("Filter cost (success path)"); ax[2].legend()
Jv= J2/J1
fig.suptitle(f"Trotter-step sweep, N={N}, L={SWEEP_L}, J2/J1 = {Jv}, Trotter order = 2")
fig.tight_layout(); fig.savefig("12trotter_sweep.png", dpi=150); plt.close(fig)
print("saved trotter_sweep.csv, trotter_sweep.png")


